# Fine-tune ARBITER — step by step

Choose **Typed (5,410 train / 590 validation)** or **80k (80,000 / 8,000)**, fine-tune a separate copy of ARBITER, and try it against the base model.

**Run one cell at a time with Shift + Enter.** Every action cell runs when you execute it: there are no hidden execution switches. Stop before Step 5 if you only want to prepare data. Training does not start in Steps 1–4. **Run All will start training.**

Use the Python kernel from your project virtual environment. All paths below are relative to the ARBITER repository.

| Step | Action |
|---|---|
| 1 | Check the environment |
| 2 | Choose the run name and settings |
| 3 | Download and build the dataset |
| 4 | Check all data before training |
| 5 | Start or resume training |
| 6 | Compare answers with the base model |

Need terminal commands instead? See the [command-line guide](../finetuning/README.md).

## 1. Check the environment

If needed, run `python -m pip install -e '.[finetuning,benchmark]'` in a terminal at the repository root, then select that environment as your notebook kernel.

The next cell checks your setup without loading model weights.

In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime
from importlib.util import find_spec
from html import escape
import gc
import json
import math
import shlex
import subprocess
import sys
from IPython.display import HTML, Markdown, display

ROOT = next((d for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (d / "finetuning/train.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open the notebook from the ARBITER repository or its finetuning/ directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def table(headers, rows):
    head = "".join(f"<th>{escape(str(h))}</th>" for h in headers)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(v))}</td>" for v in row) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

def launch(command):
    print(shlex.join(command))
    subprocess.run(command, cwd=ROOT, check=True)

print("Repository:", ROOT)
print("Python:", sys.executable)
missing = [name for name in ("torch", "transformers", "safetensors", "datasets", "tensorboard") if find_spec(name) is None]
print("Dependencies:", "OK" if not missing else "Install: " + ", ".join(missing))
if missing:
    raise RuntimeError("Install the missing dependencies, then restart the kernel.")

## 2. Choose your configuration

Select your adaptation configuration:
- **`mixture_80k`**: Balanced 80,000 decisions mixture boosted with typed & procedural reasoning (32k typed, 24k policy, 24k NLI, 0 historical replay).
- **`typed`**: Typed specialist. One observed checkpoint scored 76.45%; this is not a guaranteed outcome.

Defaults are loaded automatically from `finetuning.configs`. The base model is never overwritten.

```bash
hf download spectral-ai-lab/arbiter-272m --revision <release-commit> --local-dir checkpoints/arbiter-272m
```
Replace `<release-commit>` with the published release SHA.


In [ ]:
from finetuning.configs import get_config, AVAILABLE_CONFIGS

# Select configuration: "mixture_80k" (recommended balanced mixture) or "typed" (specialist)
CONFIG_NAME = "typed"

config = get_config(CONFIG_NAME, root_dir=ROOT)
CONFIG_NAME = config.name
PREFLIGHT_OK = False
RUN_NAME = config.run_name  # Change this for a new experiment.
DATA_DIR = config.data_dir
BASE = ROOT / "checkpoints/arbiter-272m"
TRAIN = config.train_path
VALIDATION = config.validation_path
HELDOUT = config.heldout
OUTPUT = ROOT / "results" / RUN_NAME

DEVICE = "cuda"
EPOCHS = config.epochs
BATCH_SIZE = config.batch_size
ACCUMULATION = config.accumulation
MAX_LENGTH = config.max_length
BACKBONE_LR = config.backbone_lr
HEAD_LR = config.head_lr
CLIP = config.clip
EVAL_EVERY = config.eval_every
SEED = config.seed
DETERMINISTIC = config.deterministic

assert all(v > 0 for v in (EPOCHS, BATCH_SIZE, ACCUMULATION, MAX_LENGTH, EVAL_EVERY, CLIP))
assert OUTPUT.resolve() != BASE.resolve(), "The output directory must differ from the base directory."
table(["Resource", "Path", "Available"],
      [("Config name", CONFIG_NAME, "Selected"),
       ("Base", BASE, "Yes" if BASE.exists() else "No"),
       ("Train", TRAIN, "Yes" if TRAIN.exists() else "Not yet (Step 3)"),
       ("Validation", VALIDATION, "Yes" if VALIDATION.exists() else "Not yet (Step 3)"),
       *[(f"Held-out test {i+1}", path, "Yes" if path.exists() else "No") for i, path in enumerate(HELDOUT)]])
print("Effective batch size:", BATCH_SIZE * ACCUMULATION)
print("Output destination:", OUTPUT)

required = [BASE / name for name in ("model.safetensors", "config.json", "tokenizer.json", "tokenizer_config.json")]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing release files: " + ", ".join(missing))
print("Preset:", CONFIG_NAME, "Epochs:", EPOCHS, "LRs:", BACKBONE_LR, HEAD_LR)

def run_signature():
    return repr((CONFIG_NAME, str(BASE), str(TRAIN), str(VALIDATION), tuple(map(str, HELDOUT)), str(OUTPUT), DEVICE, EPOCHS, BATCH_SIZE, ACCUMULATION, MAX_LENGTH, BACKBONE_LR, HEAD_LR, CLIP, EVAL_EVERY, SEED, DETERMINISTIC))


## 3. Build or prepare the dataset

**Run the next cell to build or prepare the data on CPU.** Existing matching outputs are verified and reused.

| Group (mixture_80k) | Training decisions |
|---|---:|
| Typed decisions and procedural workflows | 32,000 |
| Procedural policy decisions | 24,000 |
| NLI in 15 languages | 24,000 |
| Historical benchmark replay | 0 (removed) |
| **Total** | **80,000** |

An additional **8,000 decisions** are reserved for validation. For `typed`, official data (5,410 train, 590 val) is loaded from `results/adaptation/typed/`.

**Expected output:** `train.jsonl`, `validation.jsonl` and `manifest.json` (80k) or `source_manifest.json` (Typed) in `DATA_DIR`.

In [ ]:
launch([sys.executable, "scripts/prepare_adaptation.py", "--preset", CONFIG_NAME,
        "--model", str(BASE), "--heldout", *map(str, HELDOUT),
        "--output", str(DATA_DIR), "--seed", str(SEED), "--max-length", str(MAX_LENGTH)])
manifest_path = DATA_DIR / ("manifest.json" if CONFIG_NAME == "mixture_80k" else "source_manifest.json")
info = json.loads(manifest_path.read_text())
table(["Split", "Decisions"], [(split, sum(c.values()) if isinstance(c, dict) else c)
      for split, c in info["counts"].items()])


### Read one example and its expected answer

This helps you check what the model will learn. For probabilistic labels, the marked answer is the highest-probability target; the full distribution is used for training.

In [ ]:
with TRAIN.open() as f:
    example = json.loads(next(f))
print("Context:", example["context"])
print("Question:", example["instruction"])
print("Type:", example["question_type"])
expected = max(example["target"], key=example["target"].get)
table(["Option", "Description", "Target probability", "Expected"],
      [(k, v, f'{example["target"][k]:.1%}', "✓" if k == expected else "")
       for k, v in example["criteria"].items()])

## 4. Check the data before training

This checks the splits and tokenizes **all training and validation examples on CPU**. It can take time. No training happens.

**Wait for “Preflight passed” before continuing.** If this cell fails, fix the reported issue and rerun it.

In [ ]:
common = [sys.executable, "-m", "finetuning.train", "--model", str(BASE),
          "--train", str(TRAIN), "--validation", str(VALIDATION), "--heldout", *map(str, HELDOUT)]
settings = ["--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
            "--accumulation", str(ACCUMULATION), "--max-length", str(MAX_LENGTH),
            "--backbone-lr", str(BACKBONE_LR), "--head-lr", str(HEAD_LR),
            "--clip", str(CLIP), "--eval-every", str(EVAL_EVERY), "--seed", str(SEED)]
if DETERMINISTIC:
    settings += ["--deterministic"]
preflight_output = ROOT / "results/adaptation" / ("preflight-" + datetime.now().strftime("%Y%m%d-%H%M%S-%f"))
PREFLIGHT_OK = False
launch(common + settings + ["--output", str(preflight_output), "--validate-only"])
PREFLIGHT_SIGNATURE = run_signature()
PREFLIGHT_OK = True
print("Preflight passed. You can start training in Step 5.")

## 5. Train the model

**Executing the training cell starts GPU training.** The kernel stays busy until it finishes. There are 2,500 optimizer updates with the default 80k configuration.

Before starting, run TensorBoard in a **second terminal**, using the command printed below. Open `http://localhost:6006` (or your server IP).

Start with **`Validation/macro_ce`**: lower is better, and this selects the best model. Check family accuracy and score MAE only when you need more detail. The trainer's existing metrics are unchanged.

In [ ]:
print(shlex.join(["tensorboard", "--logdir", str(OUTPUT / "tensorboard"),
                  "--host", "0.0.0.0", "--port", "6006"]))

**New run:** leave `RESUME = False`. **Interrupted run:** set it to `True` and keep the same paths, settings and code. Resume restores the latest saved training state; unsaved steps are recomputed. Do not run a second training process if one is already running.

To stop this notebook's training process, interrupt the kernel. For long runs, you can copy the printed training command into a terminal instead.

In [ ]:
RESUME = False  # Change to True only to resume this run's latest checkpoint.
if not globals().get("PREFLIGHT_OK", False):
    raise RuntimeError("Complete Step 4 successfully before training.")
if globals().get("PREFLIGHT_SIGNATURE") != run_signature():
    raise RuntimeError("Settings changed: rerun Step 4 before training.")
import torch
if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError("Training requires an available BF16-capable CUDA GPU.")
command = common + settings + ["--output", str(OUTPUT), "--device", DEVICE]
if RESUME:
    command += ["--resume"]
launch(command)
print("Training finished. Best inference package:", OUTPUT / "best")

### Where are the results?

| Location inside your run folder | Contents |
|---|---|
| `best/` | Model selected by validation; use this for inference |
| `latest.pt` | Checkpoint for resuming training |
| `tensorboard/` | Learning curves |
| `validation.jsonl` | Detailed validation results |

The step-zero base can remain the best model if fine-tuning does not improve the selection metric. Training completion alone does not demonstrate a benchmark improvement.

## 6. Try the base and adapted models

Run the next two cells after training. Each example includes its **expected answer**, so the comparison is easy to read. You can edit the examples to try your own questions.

These are demonstrations, not benchmark results. Comparison uses CPU by default so it does not occupy the training GPU.

In [ ]:
examples = [
    {"context": "Policy: escalate unresolved access failures. The account was locked yesterday, but access has been restored.",
     "instruction": "Should this access problem be escalated under the policy?",
     "question_type": "noul", "criteria": {"false": "Do not escalate.", "true": "Escalate."},
     "expected": "false"},
    {"context": "All approved invoices have been paid. Invoice A is approved.",
     "instruction": "Classify the hypothesis: Invoice A has been paid.", "question_type": "choice",
     "criteria": {"entailment": "Follows from the premise.", "neutral": "Cannot be determined.", "contradiction": "Contradicts the premise."},
     "expected": "entailment"},
    {"context": "Rubric: low = no affected users; medium = 1 to 9; high = 10 or more. Four users cannot log in.",
     "instruction": "Choose the severity according to the rubric.", "question_type": "score",
     "criteria": {"low": "No users.", "medium": "1 to 9 users.", "high": "10 or more users."},
     "expected": "medium"},
]

In [ ]:
COMPARE_DEVICE = "cpu"  # "cuda" if your GPU is available.
from arbiter import ArbiterModel
packages = [("Base", BASE), ("Adapted", OUTPUT / "best")]
unavailable = [str(path) for _, path in packages if not (path / "model.safetensors").is_file()]
if unavailable:
    print("Missing packages:", ", ".join(unavailable))
else:
    comparison = []
    requests = [{k:v for k,v in e.items() if k != "expected"} for e in examples]
    for name, package in packages:
        model = ArbiterModel.from_pretrained(package, device=COMPARE_DEVICE)
        for i, (example, result) in enumerate(zip(examples, model.decide_batch(requests))):
            key = result["selected_key"]
            comparison.append([i+1, name, example["expected"], key,
                               "✓" if key == example["expected"] else "Incorrect",
                               f'{result["probabilities"][key]:.1%}'])
        del model
        gc.collect()
        if COMPARE_DEVICE.startswith("cuda"):
            import torch
            torch.cuda.empty_cache()
    table(["Example", "Model", "Expected", "Predicted", "Result", "Selected probability"], comparison)

## Optional: run the full before/after benchmarks

For a reproducible evaluation, follow the **Evaluate** section in the [CLI guide](../finetuning/README.md). It covers the historical and public suites for both models. Keep the test sets fixed and report gains as well as regressions.

## Help

| Problem | What to do |
|---|---|
| Missing base model | Download the package using the command in Step 2 |
| Nonempty output directory | Use a new `RUN_NAME`, or resume the existing compatible run |
| CUDA out of memory | For a new run, try `BATCH_SIZE = 2` and `ACCUMULATION = 16` |
| Data or length check fails | Read the error and fix the data/settings before training |
| Resume manifest mismatch | Restore the exact original code, settings and data |
| Need your own dataset | Follow the schema in the [fine-tuning plan](../docs/targeted_finetuning.md) |

[Validation report](../docs/finetuning_validation.md) · [Dataset details](../docs/build_80k.md) · [Terminal instructions](../finetuning/README.md)